In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), 'exp', sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [9]:
e.add_grp('clf', 'exp', parent_grp = None, edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', 'pipe', parent_grp = None, method = 'transform')

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})
e.build()

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
✅ Build complete!


In [11]:
e.rename_grp('preprocessor', 'preproc')

In [12]:
e.rename_grp('preproc', 'preprocessor')

In [13]:
e.build()

🔄 Building 0 node(s)
✅ Build complete!


In [14]:
e.build(rebuild=True)

🔄 Building 2 node(s)
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
✅ Build complete!


In [15]:
from analyzer import Stacker

In [16]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression)

In [17]:
from modeler import col
e.set_node('lr1', 'lr', edges = [('std', None)])
e.set_node('lr2', 'lr', edges = [('std', None), ('ohe', col.ohe_drop_first)])

In [18]:
e.build()

🔄 Building 0 node(s)
✅ Build complete!


In [19]:
e.exp()

🔄 Experimenting 2 node(s)
0 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
1 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
2 fold
  ├─ Experimenting 'lr2'...
  ├─ Experimenting 'lr1'...
✅ Experimentation complete!


In [20]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [21]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [22]:
e.add_grp('dim_reduction', parent_grp = 'preprocessor')

In [23]:
for i in e.nodes['lr1'].adhoc_idx(0, ['object', 'output', 'coef']):
    print(i)

{'spec': {'build_id': '121b4324-5d45-4d44-826d-16c0bc787c7b', 'fit_time': 0.008953332901000977, 'train_shape': (3564, 7), 'train_v_shape': (396, 7)}, 'object': LogisticRegression(), 'output_train': (<modeler._data_wrapper.PandasWrapper object at 0x7f910ddb3d10>, <modeler._data_wrapper.PandasWrapper object at 0x7f910ddb2e10>), 'output_valid': <modeler._data_wrapper.PandasWrapper object at 0x7f910ddb3aa0>, 'coef':    std__annual_income  std__debt_to_income_ratio  std__credit_score  \
0           -0.083219                  -0.722303           0.771684   

   std__loan_amount  std__interest_rate  std__grade_subgrade_no  intercept  
0         -0.030473             0.02128                0.087703   1.656942  }


In [24]:
for i in e.nodes['lr1'].experiment(0, ['output', 'coef']):
    print(i)

{'spec': {'build_id': '40a18bc1-67e4-40e7-9c42-7a636a667c8f', 'fit_time': 0.0035266876220703125, 'train_shape': (3564, 7), 'train_v_shape': (396, 7)}, 'output_train': (<modeler._data_wrapper.PandasWrapper object at 0x7f910dcef290>, <modeler._data_wrapper.PandasWrapper object at 0x7f910c3d21e0>), 'output_valid': <modeler._data_wrapper.PandasWrapper object at 0x7f910c3d0260>}


In [25]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

In [26]:
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.build()

  └─ Effeced 3 dependent node(s): ['lr1', 'lr2', 'pca']
🔄 Building 1 node(s)
  ├─ Building 'std'...
  ├─ Building 'std'...
  ├─ Building 'std'...
✅ Build complete!


In [27]:
e.exp('lr*', retry=True)

🔄 Experimenting 2 node(s)
0 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
1 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
2 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
✅ Experimentation complete!


In [ ]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

In [ ]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [ ]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

In [ ]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

In [ ]:
from analyzer import Stacker
s = Stacker(e, target_edge = (None, [y]), output_var = slice(0, -1))

In [ ]:
from modeler._metric import Metric
from sklearn.metrics import roc_auc_score
m = Metric('AUC', e, target_edge = (None, [y]), output_var = slice(0, -1), metric_func = roc_auc_score, include_train = True)

In [ ]:
pd.concat([
    m.get_metric(0, 'lr1'),
    m.get_metric(1, 'lr1')
], axis=0).rename('lr1').to_frame().T

In [ ]:
from modeler._stacker import Stacker
s = Stacker(e, target_edge = (None, [y]), output_var = slice(0, -1))

In [ ]:
s.get_stack(2, 'lr1')

In [ ]:
e.add_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [ ]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params = {'cat_features': X_cat})

In [ ]:
import lightgbm as lgb

In [ ]:
e.add_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [ ]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

In [ ]:
e.build()

In [ ]:
e.desc_node_vars('cb1', 0)

In [ ]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_['validation_1']

In [ ]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

In [ ]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

In [ ]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

In [ ]:
e._find_descendants('std')

In [ ]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [ ]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

In [ ]:
from modeler import create_like

In [ ]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

In [ ]:
e2.build()

In [ ]:
e2.nodes['lgb1'].get_result("evals_result")

In [ ]:
for (true_train, true_train_v), (prd_train, prd_train_v) in  zip(
    e.get_node_train_output(0, None, [y]),
    e.get_node_train_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_train.data, prd_train.data), 
        roc_auc_score(true_train_v.data, prd_train_v.data)
    )

In [ ]:
for true_valid, prd_valid in  zip(
    e.get_node_valid_output(0, None, [y]),
    e.get_node_valid_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_valid.data, prd_valid.data)
    )

In [ ]:
class Metric:
    def __init__(
        self, e, target_edge, output_var, metric_func, include_train = False
    ):
        self.e = e
        self.target_edge = target_edge
        self.output_var = output_var
        self.include_train = include_train
        self.metric_func = metric_func
        self.result = {}
        self.build_ids = {}

    def calc_idx(self, nodes, idx):
        result = {}
        grps = {}
        if self.include_train:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and (node, idx) in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_output(idx, None, [y]), self.e.get_node_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for no, ((true_train, true_valid), (prd_train, prd_valid)) in enumerate(iterator):
                    result_train = self.metric_func(true_train[0].data, prd_train[0].data)
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)
                    if true_train[1] is not None:
                        result_sub = {
                            (idx, 'train', f'train_{no}'): result_train,
                            (idx, 'train', f'valid_{no}'): self.metric_func(true_train[1].data, prd_train[1].data),
                            (idx, 'valid', ''): result_valid
                        }
                    else:
                        result_sub = {
                            (idx, 'train'): result_train, (idx, 'valid'): result_valid
                        }
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)
        else:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and node in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_valid_output(idx, None, [y]), self.e.get_node_valid_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for true_valid, prd_valid in iterator:
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)                    
                    result_sub = {idx: result_valid}
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)

        c, mx = None, -1
        for i in grps.values():
            i = i[::-1]
            if c is None:
                c = i
            else:
                mx = max(mx, len(i))
                for j in range(min(len(c), len(i))):
                    if c[j] != i[j]:
                        c = i[:j]
                        break
            if len(c) == 0:
                break
        for k, i in grps.items():
            i = tuple([''] * (mx - len(i) - len(c)) +  i[:-len(c)] + [k])
            result[k] = result[k].rename(i)
        return pd.DataFrame(result.values())

    def calc(self, nodes):
        result = [self.calc_idx(nodes, i) for i in range(self.e.get_n_splits())]
        return pd.concat(result, axis=1)

In [ ]:
for (y_true_train_t, y_true_valid_t), y_true_valid in e2.get_node_output(0, 'cb1'):
    print(y_true_train_t)
    print(y_true_valid_t)

In [ ]:
e.nodes['cb1'].objs_[0][0][0].X_

In [ ]:
import analyzer
from analyzer import Metric
importlib.reload(analyzer)

In [ ]:
m = Metric(e, (None, [y]), slice(-1, None), roc_auc_score, True)
m.set_nodes('clf')
result = m.get_metric()
result

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = None, splitter_params = {'y': y})

In [ ]:
df_input, df_output = e2.desc_node_vars('lr3', 0)
display(df_input)
df_output

In [ ]:
from analyzer import Stacker
s = Stacker(e3, (None, [y]), slice(-1, None))

In [ ]:
s.set_nodes('cb')
s.set_nodes('lr')
s.get_dataset().data

In [ ]:
e3.nodes['lr3'].objs_[0][0][0].obj.classes_

In [ ]:
lr_a.set_nodes('lr')

In [ ]:
for inner_idx, df in lr_a.result[('lr1', 0)].items():
    print(type(df['intercept']) == pd.Series, df['intercept'].to_frame().columns)

In [ ]:
lr_a.get_coef('lr1').T.groupby(level = [2]).mean().T

In [ ]:
lr_a.get_intercept('lr1').T.groupby(level = [2]).mean().T

In [ ]:
e.root.data

In [ ]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

In [ ]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

In [ ]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123), splitter_params = {'y': y})

In [ ]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
e3.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [ ]:
e3.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e3.set_node('lr1', 'lr')

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [ ]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [ ]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

In [ ]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e.set_node('lr1', 'lr')

In [ ]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

In [ ]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [ ]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

In [ ]:
df_train.to_pandas()[[y]].shape

In [ ]:
e.nodes['lr1'].y